In [1]:
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np



In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [3]:
from src.improved_model import CNN_2

# Instantiate the model
model = CNN_2().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))
# optimizer = optim.SGD(model.parameters(), lr=float(1e-1))

In [ ]:
from src.training import get_average_separation

# Training loop
num_epochs: int = 500
target_accuracy: float = .95
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.layer2.bias.data.clamp_(min=1.0)
        #model.layer2.bias.data.clamp_(min=-2.19722)
    # separation: float = get_average_separation(test_data, model)

    # Assess progress:
    # data: np.ndarray = get_intermediate_outputs_as_numpy(model, train_dataloader)
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.set_scramble_distance(min(model.scramble_distance + .05, maximum_scramble_distance))
        print(f"Scramble distance: {model.scramble_distance:.2f}")
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.1055
Epoch [2/500], Accuracy: 0.9353
Scramble distance: 0.05
Epoch [3/500], Accuracy: 0.9516
Scramble distance: 0.10
Epoch [4/500], Accuracy: 0.9588
Scramble distance: 0.15
Epoch [5/500], Accuracy: 0.9613
Scramble distance: 0.20
Epoch [6/500], Accuracy: 0.9610
Scramble distance: 0.25
Epoch [7/500], Accuracy: 0.9659
Scramble distance: 0.30
Epoch [8/500], Accuracy: 0.9671
Scramble distance: 0.35
Epoch [9/500], Accuracy: 0.9659
Scramble distance: 0.40
Epoch [10/500], Accuracy: 0.9672
Scramble distance: 0.45
Epoch [11/500], Accuracy: 0.9673
Scramble distance: 0.50
Epoch [12/500], Accuracy: 0.9675
Scramble distance: 0.55
Epoch [13/500], Accuracy: 0.9669
Scramble distance: 0.60
Epoch [14/500], Accuracy: 0.9672
Scramble distance: 0.65
Epoch [15/500], Accuracy: 0.9673
Scramble distance: 0.70
Epoch [16/500], Accuracy: 0.9655
Scramble distance: 0.75
Epoch [17/500], Accuracy: 0.9684
Scramble distance: 0.80
Epoch [18/500], Accuracy: 0.9662
Scramble distance: 0.85
Epoch [

In [ ]:
def get_signed_accuracy(model: CNN_2, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy
test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]
#print(model.float_to_binary_layer(d))

print(model.second_layer(model.float_to_binary_layer(d)))
#
# # model(test_data[0,...])
print(get_signed_accuracy(model, val_dataloader))


In [ ]:
from torch import softmax

model.eval_mode()
model.layer2.scale
#1 + softmax(model.layer2.bias, dim=0)

In [ ]:
save_model(model, "convnet_v2")